# Setup


In [ ]:

%pip install -q -U crewai crewai-tools


# Config


In [ ]:

from google.colab import userdata
import os

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY in Colab Secrets.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("API key loaded successfully.")


# Imports & LLM


In [ ]:

from crewai import Agent, Task, Crew, Process, LLM
from crewai.flow.flow import Flow, start, listen, and_
from crewai_tools import ScrapeWebsiteTool
from pydantic import BaseModel, Field
from pathlib import Path
from urllib.parse import urlparse
import json

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.4
)

OUTPUT_DIR = Path("/content/content_repurposing_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup completed.")


# Flow state


In [ ]:

class RepurposingState(BaseModel):
    source: str = ""
    article_text: str = ""
    article_points: str = ""
    linkedin_post: str = ""
    twitter_thread: str = ""
    newsletter: str = ""
    output_files: dict = Field(default_factory=dict)



from pathlib import Path
from crewai_tools import ScrapeWebsiteTool

def read_article(source):
    source = source.strip()

    if not source:
        raise ValueError("Please provide article text, a file path, or a URL.")

    # If the input is a URL, scrape the webpage.
    if source.startswith(("http://", "https://")):
        scraper = ScrapeWebsiteTool(website_url=source)
        content = scraper.run()

        if not content or not str(content).strip():
            raise ValueError("Could not extract readable content from this URL.")

        return str(content)

    # If it looks like article text, return it directly.
    if "\n" in source or len(source) > 250:
        return source

    # Otherwise, try treating it as a file path.
    try:
        path = Path(source)
        if path.is_file():
            return path.read_text(encoding="utf-8")
    except (OSError, ValueError):
        pass

    # Short input that isn't a file path is treated as article text.
    return source

print("read_article() updated successfully.")


# Social crews


In [ ]:

def create_linkedin_crew():
    agent = Agent(
        role="Professional Post Author",
        goal="Draft an impactful LinkedIn post of approximately 150 words.",
        backstory="You specialize in professional, engaging social content.",
        llm=llm,
        verbose=True
    )

    task = Task(
        description=(
            "Using these article points:\n{article_points}\n\n"
            "Write a LinkedIn post of approximately 150 words. Start with a "
            "strong hook, use readable paragraphs, preserve the facts, and "
            "finish with a relevant question or call to action."
        ),
        expected_output="One polished LinkedIn post of approximately 150 words.",
        agent=agent
    )

    return Crew(
        agents=[agent], tasks=[task],
        process=Process.sequential, verbose=True
    )


def create_twitter_crew():
    agent = Agent(
        role="Social Thread Writer",
        goal="Create an informative six-tweet thread summarizing the article.",
        backstory="You turn complex ideas into concise, readable tweets.",
        llm=llm,
        verbose=True
    )

    task = Task(
        description=(
            "Using these article points:\n{article_points}\n\n"
            "Write exactly six numbered tweets. Tweet 1 must hook the reader. "
            "Arrange the ideas logically and finish with a useful takeaway. "
            "Keep tweets concise and do not invent facts."
        ),
        expected_output="Exactly six numbered tweets, from 1/6 to 6/6.",
        agent=agent
    )

    return Crew(
        agents=[agent], tasks=[task],
        process=Process.sequential, verbose=True
    )


def create_newsletter_crew():
    agent = Agent(
        role="Newsletter Editor",
        goal="Write a concise newsletter introduction and three actionable bullet points.",
        backstory="You write concise newsletters that highlight useful insights.",
        llm=llm,
        verbose=True
    )

    task = Task(
        description=(
            "Using these article points:\n{article_points}\n\n"
            "Write a short newsletter introduction followed by exactly three "
            "informative bullet points. Preserve the source meaning and do "
            "not invent facts."
        ),
        expected_output="A newsletter introduction and exactly three bullets.",
        agent=agent
    )

    return Crew(
        agents=[agent], tasks=[task],
        process=Process.sequential, verbose=True
    )


print("All three Crew builders created.")


# Repurposing flow


In [ ]:

class ContentRepurposingFlow(Flow[RepurposingState]):

    @start()
    def load_article(self):
        self.state.article_text = read_article(self.state.source)

        if not self.state.article_text.strip():
            raise ValueError("The article contains no readable text.")

        print("Article loaded.")
        print("Characters:", len(self.state.article_text))

    @listen(load_article)
    def extract_points(self):
        agent = Agent(
            role="Article Synthesis Specialist",
            goal="Extract the core thesis and critical supporting points.",
            backstory="You summarize source material accurately without inventing facts.",
            llm=llm,
            verbose=True
        )

        task = Task(
            description=(
                "Extract the key ideas from this article:\n\n"
                "{article}\n\n"
                "Return a structured list of the main argument, important "
                "supporting details, and conclusion. Do not add outside facts."
            ).format(article=self.state.article_text),
            expected_output="A concise list of key ideas and the main conclusion.",
            agent=agent
        )

        result = Crew(
            agents=[agent],
            tasks=[task],
            process=Process.sequential,
            verbose=True
        ).kickoff()

        self.state.article_points = str(result)
        print("Key points extracted.")

    @listen(extract_points)
    def create_linkedin(self):
        result = create_linkedin_crew().kickoff(
            inputs={"article_points": self.state.article_points}
        )
        self.state.linkedin_post = str(result)
        print("LinkedIn Crew completed.")

    @listen(extract_points)
    def create_twitter(self):
        result = create_twitter_crew().kickoff(
            inputs={"article_points": self.state.article_points}
        )
        self.state.twitter_thread = str(result)
        print("Twitter Crew completed.")

    @listen(extract_points)
    def create_newsletter(self):
        result = create_newsletter_crew().kickoff(
            inputs={"article_points": self.state.article_points}
        )
        self.state.newsletter = str(result)
        print("Newsletter Crew completed.")

    @listen(and_(create_linkedin, create_twitter, create_newsletter))
    def publish_pack(self):
        outputs = {
            "linkedin_post.md": self.state.linkedin_post,
            "twitter_thread.md": self.state.twitter_thread,
            "newsletter.md": self.state.newsletter
        }

        for filename, content in outputs.items():
            path = OUTPUT_DIR / filename
            path.write_text(content, encoding="utf-8")
            self.state.output_files[filename] = str(path)

        summary = {
            "source": self.state.source,
            "article_points": self.state.article_points,
            "output_files": self.state.output_files
        }

        summary_path = OUTPUT_DIR / "summary.json"
        summary_path.write_text(
            json.dumps(summary, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

        print("\nAll three branches completed.")
        print("Saved files:", self.state.output_files)


# Article input & run


In [ ]:
sample_article = """
The Shift Toward Modular Monoliths in Engineering

Many engineering teams adopt microservices too early, introducing network latency,
distributed tracing complexity, and deployment overhead before product-market fit.
Recent industry architectural trends show high-growth software teams returning to modular monoliths.

A well-architected modular monolith maintains distinct domain boundaries inside a unified codebase.
Developers can refactor interfaces effortlessly, execute end-to-end tests locally in seconds,
and avoid distributed transaction failures. When a specific domain genuinely requires independent scaling,
it can be extracted cleanly into an isolated microservice.

Pragmatic architecture prioritizes shipping velocity and domain clarity over premature operational complexity.
"""

flow = ContentRepurposingFlow()
flow.state.source = sample_article

flow.kickoff()

# Verify files


In [ ]:

for filename in [
    "linkedin_post.md",
    "twitter_thread.md",
    "newsletter.md"
]:
    path = OUTPUT_DIR / filename

    print(f"\n{'=' * 15} {filename} {'=' * 15}")
    print(path.read_text(encoding="utf-8"))

print("\nOutput directory:", OUTPUT_DIR)
print("Summary:", OUTPUT_DIR / "summary.json")
